# Value Head Calibration

Re-align the value head with the current backbone after policy training.
Freezes backbone + policy head, trains only value head layers.
Target: empty_squares/81 (crude but produces Q-diversity for MCTS).

**Upload to Drive:**
1. `colorlines_selfplay_train.tar.gz`
2. `selfplay_v8_policy.pt.gz`
3. `pillar2w2_epoch_10.pt`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time
DRIVE = '/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2w2_epoch_10.pt', '/content/alphatrain/data/model.pt')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v8_policy.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')
!pip install -q numpy numba scipy

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f'Device: {device}')

from alphatrain.evaluate import load_model
net, max_score = load_model('/content/alphatrain/data/model.pt', device)

# Freeze everything EXCEPT value head
value_prefixes = ('value_conv', 'value_bn', 'value_fc1', 'value_fc2')
frozen, trainable = 0, 0
for name, param in net.named_parameters():
    if name.startswith(value_prefixes):
        param.requires_grad_(True)
        trainable += param.numel()
    else:
        param.requires_grad_(False)
        frozen += param.numel()

print(f'Frozen: {frozen:,} (backbone + policy)')
print(f'Trainable: {trainable:,} (value head only)')

In [ ]:
from alphatrain.dataset import TensorDatasetGPU

dataset = TensorDatasetGPU('/content/alphatrain/data/selfplay.pt',
                            augment=True, device=str(device))

n_val = int(len(dataset) * 0.05)
n_train = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_train, n_val],
                                   generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=65536, shuffle=True,
                          num_workers=0, collate_fn=dataset.collate)
print(f'Train: {n_train:,}, Val: {n_val:,}')

In [ ]:
# Calibrate value head: predict empty_squares/81 from backbone features
EPOCHS = 2
LR = 1e-3

optimizer = torch.optim.AdamW(
    [p for p in net.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-4)

def cross_entropy_soft(logits, targets):
    log_probs = F.log_softmax(logits, dim=-1)
    return -(targets * log_probs).sum(dim=-1).mean()

if device.type == 'cuda':
    net = net.to(memory_format=torch.channels_last)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
use_amp = scaler is not None

for epoch in range(EPOCHS):
    net.train()
    total_loss = 0
    n = 0
    t0 = time.time()

    for bi, batch in enumerate(train_loader):
        obs, _ = batch
        obs = obs.to(device)

        with torch.amp.autocast('cuda', enabled=use_amp):
            _, val_logits = net(obs)

            # Target: empty_squares / 81, scaled to max_score
            empty_frac = obs[:, 7, :, :].float().sum(dim=(1, 2)) / 81.0
            target_val = empty_frac * max_score

            # Two-hot encode into categorical bins (in float32, outside autocast)

        # Build targets in float32 (scatter_ needs matching dtypes)
        num_bins = val_logits.shape[1]
        bins = torch.linspace(0, max_score, num_bins, device=device)
        idx = torch.searchsorted(bins, target_val.clamp(0, max_score)) - 1
        idx = idx.clamp(0, num_bins - 2).long()
        frac = ((target_val - bins[idx]) / (bins[idx + 1] - bins[idx] + 1e-8)).clamp(0, 1)
        val_tgt = torch.zeros(val_logits.shape, device=device, dtype=torch.float32)
        val_tgt.scatter_(1, idx.unsqueeze(1), (1.0 - frac).unsqueeze(1))
        val_tgt.scatter_(1, (idx + 1).unsqueeze(1), frac.unsqueeze(1))

        with torch.amp.autocast('cuda', enabled=use_amp):
            loss = cross_entropy_soft(val_logits, val_tgt)

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        n += 1
        if (bi + 1) % 50 == 0:
            print(f'  [{bi+1}] val_loss={total_loss/n:.4f}', flush=True)

    print(f'Epoch {epoch+1}/{EPOCHS}: val_loss={total_loss/n:.4f} '
          f'({time.time()-t0:.0f}s)', flush=True)

# Check Q-diversity
net.eval()
with torch.no_grad():
    test = torch.randn(100, 18, 9, 9, device=device)
    _, tv = net(test)
    scores = net.predict_value(tv, max_val=max_score)
    print(f'\nValue predictions on 100 random boards:')
    print(f'  mean={scores.mean():.1f}, std={scores.std():.1f}')
    print(f'  std > 10 = Q-diversity restored')

In [ ]:
# Save calibrated model
DRIVE = '/content/drive/MyDrive/alphatrain'
ckpt = {
    'epoch': 10,
    'model': net.state_dict(),
    'max_score': max_score,
}
save_path = f'{DRIVE}/pillar2w2_epoch_10_calibrated.pt'
torch.save(ckpt, save_path)
print(f'Saved: {save_path} ({os.path.getsize(save_path)/1e6:.0f} MB)')